# Cabot Cases — Document & Sentence Vector Database with DuckDB

This notebook demonstrates:

- Loading medical case documents from a CSV file
- Storing documents and their constituent sentences in a normalized DuckDB schema
- Computing sentence-level embeddings using SentenceTransformers
- Similarity search: given a query sentence, find the most similar sentences in the corpus

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Load CSV File

In [ ]:
csv_path = '../data/case-teaching-cabot.csv'
df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} documents')
df.head(3)

## Split Documents into Sentences

A sentence is defined as a phrase delimited by a period followed by a space (`. `).  
Newlines inside the documents are normalised to spaces first.

In [ ]:
def split_into_sentences(text):
    """Split text on '. ' boundaries; strip and discard empty fragments."""
    # normalise whitespace so multi-line entries become a single stream of text
    text = re.sub(r'\s+', ' ', text).strip()
    fragments = text.split('. ')
    return [f.strip() for f in fragments if f.strip()]

# quick sanity check
sample_sentences = split_into_sentences(df['document'].iloc[0])
print(f'Document 1 → {len(sample_sentences)} sentences')
for s in sample_sentences[:3]:
    print(' •', s[:100])

## Create DuckDB Schema

In [ ]:
con = duckdb.connect('cabot.db')

con.execute("CREATE SEQUENCE IF NOT EXISTS document_seq START 1;")
con.execute("CREATE SEQUENCE IF NOT EXISTS sentence_seq START 1;")

# One row per source document
con.execute("""
CREATE TABLE IF NOT EXISTS document (
    document_id INTEGER PRIMARY KEY DEFAULT nextval('document_seq'),
    content     TEXT
);
""")

# One row per sentence, with FK back to document
con.execute("""
CREATE TABLE IF NOT EXISTS sentence (
    sentence_id INTEGER PRIMARY KEY DEFAULT nextval('sentence_seq'),
    document_id INTEGER REFERENCES document(document_id),
    content     TEXT
);
""")

# Embedding vector for each sentence
con.execute("""
CREATE TABLE IF NOT EXISTS sentence_embedding (
    sentence_id INTEGER REFERENCES sentence(sentence_id),
    embedding   DOUBLE[]
);
""")

print('Schema created.')

## Populate Tables

In [ ]:
for _, row in df.iterrows():
    doc_text = row['document']

    # Insert document and capture the generated id via RETURNING
    document_id = con.execute(
        "INSERT INTO document (content) VALUES (?) RETURNING document_id",
        [doc_text]
    ).fetchone()[0]

    # Insert each sentence linked to this document
    for sentence_text in split_into_sentences(doc_text):
        con.execute(
            "INSERT INTO sentence (document_id, content) VALUES (?, ?)",
            [document_id, sentence_text]
        )

total_docs = con.execute("SELECT COUNT(*) FROM document").fetchone()[0]
total_sents = con.execute("SELECT COUNT(*) FROM sentence").fetchone()[0]
print(f'Inserted {total_docs} documents and {total_sents} sentences.')

## Compute Sentence Embeddings

In [ ]:
# model = SentenceTransformer('all-MiniLM-L6-v2')  ## larger, more accurate, but slower
model = SentenceTransformer('paraphrase-MiniLM-L3-v2')  ## smaller, faster, slightly less accurate

sentences = con.execute("SELECT sentence_id, content FROM sentence").fetchall()

for sentence_id, content in sentences:
    embedding = model.encode(content)
    con.execute(
        "INSERT INTO sentence_embedding VALUES (?, ?)",
        [sentence_id, embedding.tolist()]
    )

print(f'Computed and stored embeddings for {len(sentences)} sentences.')

## Similarity Queries

Given a query sentence, rank all stored sentences by cosine similarity to the query embedding.

In [ ]:
def find_similar_sentences(query: str, top_k: int = 5):
    """Return the top-k most similar sentences to *query* with their source document id."""
    query_embedding = model.encode(query)

    rows = con.execute("""
        SELECT s.sentence_id, s.document_id, s.content, se.embedding
        FROM sentence_embedding se
        JOIN sentence s ON se.sentence_id = s.sentence_id
    """).fetchall()

    scores = []
    for sentence_id, document_id, content, embedding in rows:
        sim = cosine_similarity([query_embedding], [np.array(embedding)])[0][0]
        scores.append((sim, sentence_id, document_id, content))

    scores.sort(key=lambda x: x[0], reverse=True)

    results = []
    for sim, sentence_id, document_id, content in scores[:top_k]:
        results.append({
            'similarity': round(float(sim), 4),
            'sentence_id': sentence_id,
            'document_id': document_id,
            'sentence': content
        })
    return pd.DataFrame(results)

In [ ]:
find_similar_sentences('patient complains of chest pain and shortness of breath', top_k=5)

In [ ]:
find_similar_sentences('fever and vomiting after eating', top_k=5)